# Transformer-native 3D GPU bootstrap
Private project verification on Kaggle. Runs unit tests and a synthetic CUDA optimization probe; this is not SC-VAE training.

In [ ]:
import base64, hashlib, io, json, pathlib, subprocess, sys, zipfile
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '82db5a79d4311c2970b45f8037db1158b8f0b1f259aff7ac1c10f95cca53b365'
root = pathlib.Path("/kaggle/working/t3d-project")
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(root)
print("Source SHA256:", hashlib.sha256(payload).hexdigest())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "hilbertcurve>=2,<3", "trimesh>=4,<5", "Pillow>=10", "opencv-python-headless"], check=True)
import torch
assert torch.cuda.is_available(), "GPU allocation required; refusing CPU fallback"
print("PyTorch", torch.__version__)
print("CUDA devices", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


In [ ]:
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                        cwd=root, text=True, capture_output=True)
(root.parent/"unit-tests.log").write_text(result.stdout+result.stderr)
print(result.stdout+result.stderr)
assert result.returncode == 0, "Project tests failed"
subprocess.run([sys.executable, "-m", "tools.roundtrip_demo"], cwd=root, check=True)


In [ ]:
torch.manual_seed(42)
device = torch.device("cuda:0")
x = torch.randn(256, 32, device=device)
target = torch.randn(256, 8, device=device)
model = torch.nn.Sequential(torch.nn.Linear(32, 64), torch.nn.GELU(), torch.nn.Linear(64, 8)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.005)
losses = []
for step in range(30):
    optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.mse_loss(model(x), target)
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
    optimizer.step()
    losses.append(loss.item())
torch.cuda.synchronize()
assert losses[-1] < losses[0]
report = dict(source_sha256=hashlib.sha256(payload).hexdigest(), torch=torch.__version__,
              cuda=torch.version.cuda, devices=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
              unit_tests_passed=True, probe="synthetic MLP CUDA optimization, not SC-VAE training",
              initial_loss=losses[0], final_loss=losses[-1], steps=len(losses))
(root.parent/"gpu-report.json").write_text(json.dumps(report, indent=2))
torch.save(model.state_dict(), root.parent/"synthetic-probe.pt")
print(json.dumps(report, indent=2))


In [ ]:
subprocess.run([sys.executable, "-m", "tools.train_compression_probe"], cwd=root, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "tools.benchmark_extraction", "--source-sha256",
                hashlib.sha256(payload).hexdigest()], cwd=root, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "tools.probe_cumesh", "--source-sha256",
                hashlib.sha256(payload).hexdigest()], cwd=root, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "tools.probe_export", "--source-sha256",
                hashlib.sha256(payload).hexdigest()], cwd=root, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "tools.train_corpus_probe", "--render-weight", "0.1"], cwd=root, check=True)
